# Generate QA pairs from common EM-DAT / WWA facts

This notebook creates a benchmark of question/answer pairs using the columns:

- `common_facts_summary`
- `merge_summary`

The goal is to generate questions whose answers are grounded in information that appears in both the structured EM-DAT row and the WWA textual context.

The notebook uses Ollama locally. Make sure Ollama is running before executing the generation cells.


In [ ]:
# Optional installs if needed
# !pip install -q pandas openpyxl requests tqdm

In [1]:
from pathlib import Path
import json
import re
import time
import uuid
from typing import Any, Dict, List, Optional

import pandas as pd
import requests
from tqdm.auto import tqdm

## Configuration

Set `INPUT_FILE` to your updated Excel or CSV file. The notebook will also try to auto-detect a file containing the required columns if `INPUT_FILE` does not exist.


In [16]:
# Input file with common_facts_summary and merge_summary columns.
INPUT_FILE = "emdat_wwa_alignment_common_facts.xlsx"

# Ollama settings.
OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "gpt-oss:120b-cloud"  # change this to your local model, e.g. "gpt-oss:120b-cloud"
TEMPERATURE = 0.4

# Generation settings.
QUESTIONS_PER_ROW = 2
MAX_CONTEXT_CHARS = 12000
MAX_ROWS = None  # set to an integer for testing, e.g. 5
REQUEST_SLEEP_SECONDS = 0.2

# Output files.
OUTPUT_CSV = "common_facts_qa_benchmark.csv"
OUTPUT_JSONL = "common_facts_qa_benchmark.jsonl"
OUTPUT_XLSX = "common_facts_qa_benchmark.xlsx"
RAW_OUTPUT_JSONL = "common_facts_qa_raw_outputs.jsonl"

In [17]:
REQUIRED_COLUMNS = ["common_facts_summary", "merge_summary"]


def find_input_file(preferred: str) -> Path:
    preferred_path = Path(preferred)
    if preferred_path.exists():
        return preferred_path

    candidates = list(Path.cwd().glob("*.xlsx")) + list(Path.cwd().glob("*.csv"))
    for path in candidates:
        try:
            if path.suffix.lower() == ".csv":
                sample = pd.read_csv(path, nrows=5)
            else:
                sample = pd.read_excel(path, nrows=5)
            if all(col in sample.columns for col in REQUIRED_COLUMNS):
                return path
        except Exception:
            continue

    raise FileNotFoundError(
        f"Could not find {preferred!r} and no local .xlsx/.csv file contains columns {REQUIRED_COLUMNS}."
    )


input_path = find_input_file(INPUT_FILE)
print(f"Using input file: {input_path}")

if input_path.suffix.lower() == ".csv":
    df = pd.read_csv(input_path)
else:
    df = pd.read_excel(input_path)

missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if MAX_ROWS is not None:
    df = df.head(MAX_ROWS).copy()

print(df.shape)
df.head()

Using input file: emdat_wwa_alignment_common_facts.xlsx
(57, 59)


,event_id,Historic,Classification Key,Disaster Group,Disaster Subgroup,Disaster Type,Disaster Subtype,External IDs,Event Name,ISO,...,same_event,same_event_confidence,common_facts_json,common_facts_summary,structured_only_facts_json,text_only_facts_json,merge_summary,alignment_status,alignment_error,ollama_raw_response
0,2014-9025-BRA,No,nat-cli-dro-dro,Natural,Climatological,Drought,Drought,NaN,NaN,BRA,...,0.0,low,"[{""fact_type"": ""hazard"", ""canonical_fact"": ""Dr...",[hazard; high] Drought in Brazil | [date; high...,"[{""fact"": ""Event ID 2014-9025-BRA"", ""structure...","[{""fact"": ""Severe water shortages in São Paulo...",Both sources describe a drought event in Brazi...,ok,NaN,"{\n ""same_event"": false,\n ""same_event_confi..."
1,2015-0525-GBR,No,nat-met-sto-sto,Natural,Meteorological,Storm,Storm (General),HANZE:2150,Storm Desmond (Ted),GBR,...,1.0,high,"[{""fact_type"": ""location"", ""canonical_fact"": ""...","[location; high] Cumbria, United Kingdom | [da...","[{""fact"": ""Total Deaths: 3"", ""structured_evide...","[{""fact"": ""Record rainfall of 13.44 inches (34...",Both sources refer to the same extreme weather...,ok,NaN,"{\n ""same_event"": true,\n ""same_event_confid..."
2,2015-0504-IND,No,nat-hyd-flo-flo,Natural,Hydrological,Flood,Flood (General),DFO:4309|GLIDE:TC-2015-000163,NaN,IND,...,1.0,high,"[{""fact_type"": ""location"", ""canonical_fact"": ""...","[location; high] Chennai, India | [date; high]...","[{""fact"": ""Historic: No"", ""structured_evidence...","[{""fact"": ""Heaviest one‑day rainfall in over a...",Both sources describe the December 2015 flood ...,ok,NaN,"{\n ""same_event"": true,\n ""same_event_confid..."
3,2015-9616-SOM,No,nat-cli-dro-dro,Natural,Climatological,Drought,Drought,GLIDE:DR-2015-000134,NaN,SOM,...,1.0,high,"[{""fact_type"": ""location"", ""canonical_fact"": ""...",[location; high] Somalia | [disaster_type; hig...,"[{""fact"": ""Event ID 2015-9616-SOM"", ""structure...","[{""fact"": ""La Niña increased dry‑season probab...",Both sources describe the same 2015‑2017 Somal...,ok,NaN,"{\n ""same_event"": true,\n ""same_event_confid..."
4,2015-9545-ETH,No,nat-cli-dro-dro,Natural,Climatological,Drought,Drought,GLIDE:DR-2015-000109,NaN,ETH,...,1.0,high,"[{""fact_type"": ""location"", ""canonical_fact"": ""...",[location; high] Ethiopia | [hazard; high] Dro...,"[{""fact"": ""Event ID: 2015-9545-ETH"", ""structur...","[{""fact"": ""The drought was described as the wo...",Both sources describe the same 2015 Ethiopian ...,ok,NaN,"{\n ""same_event"": true,\n ""same_event_confid..."


## Helpers for row context and JSON parsing

In [18]:
def clean_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def safe_json_loads(text: str) -> Optional[Any]:
    try:
        return json.loads(text)
    except Exception:
        return None


def extract_json_array(text: str) -> Optional[List[Dict[str, Any]]]:
    """Parse a JSON array from an LLM response, with fallbacks for fenced code blocks."""
    if not isinstance(text, str):
        return None

    stripped = text.strip()

    # Remove markdown fences if present.
    fence = re.search(r"```(?:json)?\s*(.*?)```", stripped, flags=re.DOTALL | re.IGNORECASE)
    if fence:
        stripped = fence.group(1).strip()

    parsed = safe_json_loads(stripped)
    if isinstance(parsed, list):
        return parsed
    if isinstance(parsed, dict) and isinstance(parsed.get("items"), list):
        return parsed["items"]
    if isinstance(parsed, dict) and isinstance(parsed.get("questions"), list):
        return parsed["questions"]

    # Fallback: extract the first JSON array.
    match = re.search(r"\[.*\]", stripped, flags=re.DOTALL)
    if match:
        parsed = safe_json_loads(match.group(0))
        if isinstance(parsed, list):
            return parsed

    return None


def compact_row_metadata(row: pd.Series) -> Dict[str, str]:
    """Collect useful non-long metadata from the row."""
    preferred_cols = [
        "DisNo.", "disno", "event_id", "country", "Country", "ISO", "Start Year", "Start Month",
        "Start Day", "End Year", "End Month", "End Day", "Disaster Group", "Disaster Subgroup",
        "Disaster Type", "Disaster Subtype", "Event Name", "Location", "url", "source_url"
    ]
    metadata = {}
    for col in preferred_cols:
        if col in row.index:
            val = clean_text(row[col])
            if val:
                metadata[col] = val
    return metadata


def build_generation_context(row: pd.Series, row_index: int) -> str:
    metadata = compact_row_metadata(row)
    common = clean_text(row.get("common_facts_summary", ""))
    merge = clean_text(row.get("merge_summary", ""))

    optional_parts = []
    for col in ["common_facts_json", "structured_only_facts_json", "text_only_facts_json"]:
        if col in row.index:
            val = clean_text(row[col])
            if val:
                optional_parts.append(f"{col}: {val[:3000]}")

    context = f"""
ROW_INDEX: {row_index}

STRUCTURED ROW METADATA:
{json.dumps(metadata, ensure_ascii=False, indent=2)}

COMMON FACTS SUMMARY:
{common}

MERGE SUMMARY:
{merge}
""".strip()

    if optional_parts:
        context += "\n\nOPTIONAL FACT COLUMNS:\n" + "\n".join(optional_parts)

    if len(context) > MAX_CONTEXT_CHARS:
        context = context[:MAX_CONTEXT_CHARS].rstrip()

    return context


def row_has_enough_common_info(row: pd.Series) -> bool:
    common = clean_text(row.get("common_facts_summary", ""))
    merge = clean_text(row.get("merge_summary", ""))
    combined = f"{common} {merge}".strip()
    if len(combined) < 40:
        return False
    bad_markers = ["no common", "not enough", "insufficient", "none identified"]
    lowered = combined.lower()
    return not any(marker in lowered for marker in bad_markers)

## Ollama call

In [19]:
SYSTEM_PROMPT = """
Your taks is to generate benchmark simple question-answer pairs for evaluating a hybrid KG + RAG question answering system.

The input contains facts that were identified as common between two sources:
1. structured EM-DAT-style event information;
2. textual WWA study context.

Generate questions only from facts that are explicitly supported by the common information or merge summary.
Do not create questions that require facts present in only one source.
Do not invent event details, numbers, dates, places, or attribution claims.

Return only a valid JSON array. Do not include markdown.
Each item must have this schema:
{
  "question": string,
  "answer": string,
  "answer_type": "entity" | "date" | "number" | "boolean" | "list" | "short_text",
  "required_sources": ["KG", "RAG"],
  "evidence_from_common_facts": string,
  "difficulty": "single_hop" | "multi_hop" | "hybrid",
  "reasoning": string
}

Rules:
- The answer must be concise and directly evaluable.
- Prefer factual questions about shared event identity, country, disaster type, hazard, location, date, impacts, or common descriptions.
- Include at least one question that requires combining two common facts when possible.
- For numeric answers, use only the number and unit if the unit is necessary.
- For dates, use ISO format if the date is explicit; otherwise use the exact granularity available.
- Avoid generic questions like the following: "In which month and year did the drought event begin in Brazil?".
""".strip()


def make_user_prompt(row_context: str, n_questions: int) -> str:
    return f"""
Generate {n_questions} question-answer pairs from the common information below.

The questions must be answerable from information shared by both the structured row and the WWA textual context.
The benchmark is intended to evaluate a system that has access to both sources, so required_sources must always be ["KG", "RAG"].

ROW CONTEXT:
{row_context}
""".strip()


def call_ollama(messages: List[Dict[str, str]], model: str = OLLAMA_MODEL, temperature: float = TEMPERATURE) -> str:
    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {"temperature": temperature},
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=180)
    response.raise_for_status()
    data = response.json()
    return data["message"]["content"]


def generate_qa_for_row(row: pd.Series, row_index: int) -> Dict[str, Any]:
    row_context = build_generation_context(row, row_index)

    if not row_has_enough_common_info(row):
        return {
            "row_index": row_index,
            "status": "skipped_weak_common_info",
            "raw_response": "[]",
            "items": [],
            "error": "",
        }

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(row_context, QUESTIONS_PER_ROW)},
    ]

    try:
        raw = call_ollama(messages)
        items = extract_json_array(raw)
        if items is None:
            return {
                "row_index": row_index,
                "status": "parse_error",
                "raw_response": raw,
                "items": [],
                "error": "Could not parse JSON array",
            }
        return {
            "row_index": row_index,
            "status": "ok",
            "raw_response": raw,
            "items": items,
            "error": "",
        }
    except Exception as exc:
        return {
            "row_index": row_index,
            "status": "error",
            "raw_response": "",
            "items": [],
            "error": str(exc),
        }

## Generate QA pairs

For a quick test, set `MAX_ROWS = 3` in the configuration cell and rerun from the top.

In [20]:
raw_results = []
benchmark_rows = []

for row_index, row in tqdm(df.iterrows(), total=len(df)):
    result = generate_qa_for_row(row, row_index)
    raw_results.append(result)

    metadata = compact_row_metadata(row)
    source_url = metadata.get("url") or metadata.get("source_url") or ""
    event_id = metadata.get("DisNo.") or metadata.get("disno") or metadata.get("event_id") or ""

    for item_idx, item in enumerate(result.get("items", []), start=1):
        if not isinstance(item, dict):
            continue

        question = clean_text(item.get("question", ""))
        answer = clean_text(item.get("answer", ""))
        if not question or not answer:
            continue

        required_sources = item.get("required_sources", ["KG", "RAG"])
        if isinstance(required_sources, str):
            required_sources = [required_sources]
        required_sources = [str(s).upper() for s in required_sources]
        if "KG" not in required_sources:
            required_sources.append("KG")
        if "RAG" not in required_sources:
            required_sources.append("RAG")

        qa_id = f"common_{row_index}_{item_idx}_{uuid.uuid4().hex[:8]}"
        benchmark_rows.append({
            "id": qa_id,
            "source_row_index": row_index,
            "event_id": event_id,
            "question": question,
            "gold_answer": answer,
            "answer_type": clean_text(item.get("answer_type", "short_text")) or "short_text",
            "required_sources": required_sources,
            "difficulty": clean_text(item.get("difficulty", "hybrid")) or "hybrid",
            "reasoning": clean_text(item.get("reasoning", "")),
            "evidence_from_common_facts": clean_text(item.get("evidence_from_common_facts", "")),
            "common_facts_summary": clean_text(row.get("common_facts_summary", "")),
            "merge_summary": clean_text(row.get("merge_summary", "")),
            "source_url": source_url,
            "generation_status": result.get("status", ""),
            "generation_error": result.get("error", ""),
        })

    time.sleep(REQUEST_SLEEP_SECONDS)

benchmark_df = pd.DataFrame(benchmark_rows)
print(f"Generated {len(benchmark_df)} QA pairs from {len(df)} rows.")
benchmark_df.head()

  0%|          | 0/57 [00:00<?, ?it/s]

Generated 112 QA pairs from 57 rows.


,id,source_row_index,event_id,question,gold_answer,answer_type,required_sources,difficulty,reasoning,evidence_from_common_facts,common_facts_summary,merge_summary,source_url,generation_status,generation_error
0,common_0_1_80e45c07,0,2014-9025-BRA,What type of natural hazard began in Brazil in...,Drought,entity,"[KG, RAG]",multi_hop,The system must link the location (Brazil) wit...,Both the structured KG row lists Disaster Type...,[hazard; high] Drought in Brazil | [date; high...,Both sources describe a drought event in Brazi...,https://www.worldweatherattribution.org/southe...,ok,
1,common_0_2_6cf4a420,0,2014-9025-BRA,When did the drought event in Brazil start?,2014-01,date,"[KG, RAG]",single_hop,The answer is directly stated in both sources ...,The KG row records Start Year 2014 and Start M...,[hazard; high] Drought in Brazil | [date; high...,Both sources describe a drought event in Brazi...,https://www.worldweatherattribution.org/southe...,ok,
2,common_1_1_530f3d2e,1,2015-0525-GBR,What is the name of the storm that caused floo...,Storm Desmond,entity,"[KG, RAG]",single_hop,The answer is directly stated by the shared fa...,Both sources agree on the event name (Storm De...,"[location; high] Cumbria, United Kingdom | [da...",Both sources refer to the same extreme weather...,https://www.worldweatherattribution.org/uks-st...,ok,
3,common_1_2_fd11abb1,1,2015-0525-GBR,Which region of the United Kingdom experienced...,Cumbria,entity,"[KG, RAG]",multi_hop,The system must combine the shared date range ...,The common facts specify the location as Cumbr...,"[location; high] Cumbria, United Kingdom | [da...",Both sources refer to the same extreme weather...,https://www.worldweatherattribution.org/uks-st...,ok,
4,common_2_1_d2b65ddf,2,2015-0504-IND,"What type of disaster affected Chennai, India ...",Flood,entity,"[KG, RAG]",single_hop,The answer is directly stated in the shared fa...,Both sources list the disaster type as Flood f...,"[location; high] Chennai, India | [date; high]...",Both sources describe the December 2015 flood ...,https://www.worldweatherattribution.org/chenna...,ok,


## Save outputs

In [21]:
# Save raw LLM outputs for debugging.
with open(RAW_OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for result in raw_results:
        f.write(json.dumps(result, ensure_ascii=False) + "\n")

# Save benchmark in multiple formats.
if not benchmark_df.empty:
    benchmark_df.to_csv(OUTPUT_CSV, index=False)
    benchmark_df.to_excel(OUTPUT_XLSX, index=False)
    with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
        for record in benchmark_df.to_dict(orient="records"):
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Saved files:")
print(f"- {RAW_OUTPUT_JSONL}")
print(f"- {OUTPUT_CSV}")
print(f"- {OUTPUT_XLSX}")
print(f"- {OUTPUT_JSONL}")

Saved files:
- common_facts_qa_raw_outputs.jsonl
- common_facts_qa_benchmark.csv
- common_facts_qa_benchmark.xlsx
- common_facts_qa_benchmark.jsonl


## Basic quality checks

In [ ]:
if benchmark_df.empty:
    print("No benchmark rows generated.")
else:
    print("Rows:", len(benchmark_df))
    print("Answer types:")
    print(benchmark_df["answer_type"].value_counts(dropna=False))
    print("\nDifficulty:")
    print(benchmark_df["difficulty"].value_counts(dropna=False))
    print("\nGeneration status:")
    print(benchmark_df["generation_status"].value_counts(dropna=False))

    display_cols = [
        "id", "source_row_index", "event_id", "question", "gold_answer", "answer_type",
        "difficulty", "evidence_from_common_facts", "source_url"
    ]
    display(benchmark_df[display_cols].head(10))

## Optional: inspect rows that failed or produced no output

In [ ]:
status_df = pd.DataFrame([
    {
        "row_index": r.get("row_index"),
        "status": r.get("status"),
        "error": r.get("error"),
        "n_items": len(r.get("items", [])),
        "raw_response_preview": clean_text(r.get("raw_response", ""))[:500],
    }
    for r in raw_results
])

status_df.to_csv("common_facts_qa_generation_status.csv", index=False)
status_df.head(20)